# Phenotype preprocessing

`DX_ASTATO2_2022` already encodes physician-diagnosed asthma together with allergic sensitization. Value `2` is used to define allergic asthma.


In [ ]:
from pathlib import Path
import pandas as pd

PHENO_FILE = Path("data/Pheno_Meth_input.txt")
UPDATED_CELLTYPE_FILE = Path("data/celltype_updates.csv")
OUTPUT_FILE = Path("data/Pheno_Meth_1_pheno_clean.txt")

CELL_COLS = [
    "EOSINO", "EOSINOpc", "LYMPHO", "LYMPHOpc",
    "MONO", "MONOpc", "NEUTRO", "NEUTROpc",
    "BASO", "BASOpc",
]


In [ ]:
pheno = pd.read_csv(PHENO_FILE, sep="\t")
print(pheno.shape)
pheno.head()


## Outcome definitions


In [ ]:
pheno["AD"] = (pheno["PH_DERM"] == 2).astype(int)
pheno["AA"] = (pheno["DX_ASTATO2_2022"] == 2).astype(int)

pheno["AD_only"] = ((pheno["AD"] == 1) & (pheno["AA"] == 0)).astype(int)
pheno["AA_only"] = ((pheno["AA"] == 1) & (pheno["AD"] == 0)).astype(int)
pheno["control"] = ((pheno["AA"] == 0) & (pheno["AD"] == 0)).astype(int)
pheno["AD_and_AA"] = ((pheno["AA"] == 1) & (pheno["AD"] == 1)).astype(int)

print(pheno["AA"].value_counts(dropna=False))


## Optional cell-type updates


In [ ]:
if UPDATED_CELLTYPE_FILE.exists():
    pheno_update = pd.read_csv(UPDATED_CELLTYPE_FILE)
    pheno_update = pheno_update.dropna(subset=["NEUTROpc"])
    pheno_update = pheno_update[pheno_update["NEUTROpc"] != "na"].copy()
    for col in CELL_COLS:
        if col in pheno_update.columns:
            pheno_update[col] = pd.to_numeric(pheno_update[col], errors="coerce")
    pheno = pheno.set_index("ID")
    pheno_update = pheno_update.set_index("ID")
    common_cols = [c for c in CELL_COLS if c in pheno.columns and c in pheno_update.columns]
    pheno.loc[pheno_update.index, common_cols] = pheno_update[common_cols]
    pheno = pheno.reset_index()


## Quality control and export


In [ ]:
cell_pc = [c for c in ["EOSINOpc", "LYMPHOpc", "MONOpc", "NEUTROpc", "BASOpc"] if c in pheno.columns]
pheno["total_cell_types"] = pheno[cell_pc].sum(axis=1, skipna=False)
pheno = pheno.dropna(subset=["NEUTROpc"]).copy()
pheno.to_csv(OUTPUT_FILE, sep="\t", index=False)
print(pheno.shape)
